# 01 — Corral vs 2025 XLSX diff

Quantify what the analyst's `2025 Transactions and Allocations Details.xlsx` holds that Corral cannot produce.

Read-only. No DDL. Outputs:

- `out/diff_report.json` — machine-readable counts + per-column match rates.
- `out/diff_report.md` — same, formatted for humans.

Reuses the SQL from `erd/validate_transactions_view.py` verbatim.

In [ ]:
from __future__ import annotations

import json
import re
import sys
from pathlib import Path

import pandas as pd
from sqlalchemy import text

NB_DIR = Path.cwd()
REPO_ROOT = NB_DIR.parent if NB_DIR.name == "notebooks" else NB_DIR
OUT = NB_DIR / "out" if NB_DIR.name == "notebooks" else NB_DIR / "notebooks" / "out"
OUT.mkdir(parents=True, exist_ok=True)

# Reuse the validated SQL + engine helper from erd/.
sys.path.insert(0, str(REPO_ROOT / "erd"))
from db_corral import get_engine  # noqa: E402
from validate_transactions_view import VIEW_SQL  # noqa: E402

XLSX = REPO_ROOT / "data" / "raw_data" / "2025 Transactions and Allocations Details.xlsx"
assert XLSX.exists(), f"XLSX not found: {XLSX}"
print(f"XLSX: {XLSX.name}")
print(f"Output dir: {OUT}")

## Load XLSX + Corral view

In [ ]:
xlsx = pd.read_excel(XLSX, dtype=str)
print(f"XLSX rows: {len(xlsx):,}  cols: {len(xlsx.columns)}")
print("Columns:", list(xlsx.columns))
xlsx.head(3)

In [ ]:
engine = get_engine()
with engine.connect() as conn:
    corral = pd.read_sql(text(VIEW_SQL), conn)
print(f"Corral view rows: {len(corral):,}")
print(f"Distinct TransactionID: {corral['TransactionID'].nunique():,}")
corral.head(3)

## Row-level diff

Left-join XLSX rows (that carry a synthetic `TransactionID`) to the Corral view on that key. Every XLSX row lands in one of:

- `in_both_matching` — joined, and the core columns all agree.
- `in_both_differing` — joined, but at least one core column disagrees.
- `xlsx_only` — no Corral row. These are the orphans Corral doesn't know about.

In [ ]:
def norm(x) -> str:
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return ""
    return re.sub(r"\s+", " ", str(x)).strip().lower()


xlsx_keyed = xlsx.dropna(subset=["TransactionID"]).copy()
xlsx_keyed["TransactionID"] = xlsx_keyed["TransactionID"].astype(str).str.strip()
xlsx_nokey = xlsx[xlsx["TransactionID"].isna() | (xlsx["TransactionID"].astype(str).str.strip() == "")]
corral["TransactionID"] = corral["TransactionID"].astype(str).str.strip()
corral_one = corral.sort_values("TdrTransactionID").drop_duplicates("TransactionID", keep="first")

xlsx_keyed = xlsx_keyed.rename(columns={
    "APN": "APN_xlsx",
    "Jurisdiction": "Jurisdiction_xlsx",
    "Notes": "Notes_xlsx",
    "Quantity": "Quantity_xlsx",
})
joined = xlsx_keyed.merge(corral_one, on="TransactionID", how="left")
n_joined = int(joined["TdrTransactionID"].notna().sum())
print(f"XLSX rows total: {len(xlsx):,}")
print(f"  with TransactionID: {len(xlsx_keyed):,}")
print(f"  without (orphan candidates): {len(xlsx_nokey):,}")
print(f"Join rate on TransactionID: {n_joined}/{len(xlsx_keyed)} = {n_joined/len(xlsx_keyed):.1%}")

## Column-level diff

For the joined rows, check each XLSX column against its best Corral source. Columns known to be missing from Corral are listed separately (no join attempted).

In [ ]:
pairs = [
    ("Transaction Type",               "TransactionType"),
    ("APN_xlsx",                       "APN"),
    ("Jurisdiction_xlsx",              "Jurisdiction"),
    ("Development Right",              "DevelopmentRight"),
    ("Quantity_xlsx",                  "Quantity"),
    ("Transaction Record ID",          "AccelaRecordID"),
    ("TRPA Status",                    "TRPAStatus"),
    ("TRPA Status Date",               "TRPAStatusDate"),
    ("Local Jurisdiction Project #",   "LocalJurisdictionProjectNumber"),
    ("Local Status",                   "LocalStatus"),
    ("Local Status Date",              "LocalStatusDate"),
    ("Notes_xlsx",                     "Notes"),
]

j = joined[joined["TdrTransactionID"].notna()]
rows = []
for xcol, dcol in pairs:
    xv = j[xcol].map(norm) if xcol in j.columns else pd.Series([""] * len(j))
    dv = j[dcol].map(norm) if dcol in j.columns else pd.Series([""] * len(j))
    considered = int(((xv != "") | (dv != "")).sum())
    matched = int(((xv == dv) & ((xv != "") | (dv != ""))).sum())
    rate = matched / considered if considered else None
    rows.append({"xlsx_column": xcol, "corral_column": dcol,
                 "considered": considered, "matched": matched,
                 "match_rate": round(rate, 4) if rate is not None else None})
column_match = pd.DataFrame(rows)
column_match

## Gap classifier

Classify every XLSX column into one of four buckets. The `corral_missing` + `corral_populated_poorly` set is what the transition table must carry (see notebook 02).

In [ ]:
# Classifications come from erd/xlsx_decomposition.md — category 1 (Corral-native),
# category 2 (Accela/permit workflow), category 3 (analyst unique), X (drop), derived.
# Here we collapse into the four labels the transition table cares about.
CLASSIFICATION = {
    "TransactionID":                "derived",
    "Transaction Type":             "corral_native",
    "APN":                          "corral_native",
    "Jurisdiction":                 "corral_native",
    "Development Right":            "corral_native",
    "Allocation Number":            "corral_missing",
    "Quantity":                     "corral_native",
    "Transaction Record ID":        "corral_populated_poorly",
    "Transaction Created Date":     "corral_missing",
    "Transaction Acknowledged Date":"corral_missing",
    "Development Type":             "corral_missing",
    "Detailed Development Type":    "corral_missing",
    "Status Jan 2026":              "drop",
    "TRPA/MOU Project #":           "corral_missing",
    "TRPA Status":                  "corral_populated_poorly",
    "TRPA Status Date":             "corral_populated_poorly",
    "Local Jurisdiction Project #": "corral_populated_poorly",
    "Local Status":                 "corral_populated_poorly",
    "Local Status Date":            "corral_populated_poorly",
    "Year Built":                   "corral_missing",
    "PM Year Built":                "corral_missing",
    "Notes":                        "corral_populated_poorly",
}

classification = pd.DataFrame([
    {"column": c, "classification": CLASSIFICATION.get(c, "unknown")}
    for c in xlsx.columns
])
print(classification["classification"].value_counts())
classification

## Write outputs

In [ ]:
report = {
    "xlsx_file": XLSX.name,
    "xlsx_rows": int(len(xlsx)),
    "xlsx_rows_with_transaction_id": int(len(xlsx_keyed)),
    "xlsx_rows_without_transaction_id": int(len(xlsx_nokey)),
    "corral_view_rows": int(len(corral)),
    "corral_distinct_transaction_ids": int(corral["TransactionID"].nunique()),
    "joined_rows": n_joined,
    "join_rate": round(n_joined / len(xlsx_keyed), 4) if len(xlsx_keyed) else None,
    "column_match": column_match.to_dict("records"),
    "classification": classification.to_dict("records"),
}
(OUT / "diff_report.json").write_text(json.dumps(report, indent=2, default=str), encoding="utf-8")
print(f"Wrote {OUT / 'diff_report.json'}")

In [ ]:
lines = []
lines.append(f"# Diff report — {XLSX.name} vs Corral\n")
lines.append(f"- XLSX rows: **{len(xlsx):,}** (with TransactionID: {len(xlsx_keyed):,}; without: {len(xlsx_nokey):,})")
lines.append(f"- Corral view rows: **{len(corral):,}** (distinct TransactionID: {corral['TransactionID'].nunique():,})")
lines.append(f"- Join rate on TransactionID: **{n_joined}/{len(xlsx_keyed)} = {n_joined/len(xlsx_keyed):.1%}**\n")
lines.append("## Per-column match rate (joined rows)\n")
lines.append("| XLSX column | Corral column | Considered | Matched | Rate |")
lines.append("|---|---|---:|---:|---:|")
for r in column_match.to_dict("records"):
    rate = f"{r['match_rate']:.1%}" if r["match_rate"] is not None else "n/a"
    lines.append(f"| {r['xlsx_column']} | {r['corral_column']} | {r['considered']} | {r['matched']} | {rate} |")
lines.append("\n## Classification\n")
lines.append("| Column | Bucket |")
lines.append("|---|---|")
for r in classification.to_dict("records"):
    lines.append(f"| {r['column']} | `{r['classification']}` |")
(OUT / "diff_report.md").write_text("\n".join(lines) + "\n", encoding="utf-8")
print(f"Wrote {OUT / 'diff_report.md'}")